# 01 — Aggregate Metadata

Builds the spine table: every DragonForce studio album + track, pulled from MusicBrainz.
Everything downstream (lyrics, key-change annotation, whatever audio data we salvage) joins to this table on `album` + `track_title`.

Run cells top to bottom the first time. After that, feel free to jump around — that's the whole point of notebooks.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.musicbrainz_client import get_release_groups, get_release_tracks

In [ ]:
# Pull the album list. This hits the API once per album for tracks, so it's slow
# (MusicBrainz asks for ~1 request/sec) — expect this cell to take a minute or two.
albums = get_release_groups()
albums_df = pd.DataFrame(albums)
albums_df[["title", "first-release-date", "primary-type"]].sort_values("first-release-date")

In [ ]:
# Filter to studio albums only (drop live albums, compilations, remix albums)
# before pulling tracks — check this list against the README's known discography
# and adjust the filter if MusicBrainz's "secondary-types" tagging is inconsistent.
studio_albums_df = albums_df[albums_df["secondary-types"].apply(lambda x: len(x) == 0)]
studio_albums_df[["title", "first-release-date"]]

In [ ]:
# NOTE: get_release_tracks() needs a *release* MBID (a specific pressing),
# not a release-group MBID (the abstract "album" entity). You'll need to
# pick one release per album — e.g. via album["releases"][0]["id"] once you
# fetch releases for each release-group, or look them up manually on
# musicbrainz.org for the canonical/original pressing of each album.
#
# Left as a TODO since which pressing counts as canonical is a judgment
# call worth making deliberately rather than defaulting to "whatever's first".

In [ ]:
# Once track-level data is joined in, save the spine table:
# studio_albums_df.to_csv("../data/processed/albums.csv", index=False)